In [ ]:
# Importing modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import calendar

In [ ]:
# Reading files
absent = pd.read_csv("./Absent_Data.csv")
compensation = pd.read_csv("./compensation.csv")
reasons = pd.read_csv("./Reasons.csv")

In [ ]:
# Exploring data
absent.head()
absent.info()
compensation.info()
reasons.info()

In [ ]:
# Merging absent and compensation
merge1 = absent.merge(
    compensation,
    how = "inner",
    on="ID",
    validate="1:1"
)

In [ ]:
# merging 1st merge with reasons
merge2 = merge1.merge(
    reasons,
    how="inner",
    left_on="Reason for absence",
    right_on="Number",
    indicator=True,
    validate="m:1"
)

In [ ]:
# Exploring final merge
merge2.columns
merge2.head()

In [ ]:
# Finding healthiest employees for the bonus
print(
    merge2[
        (merge2["Social drinker"] == 0) &
        (merge2["Social smoker"] == 0) &
        (merge2["Body mass index"].between(19,25)) &
        (merge2["Absenteeism time in hours"] < merge2["Absenteeism time in hours"].mean() )
    ].sort_values(["Body mass index", "Absenteeism time in hours", "ID"], ascending = [False, True, True]).head(10)
)

In [ ]:
# Defining compensation per year given total budget
def comp_per_year(budget):
    workers = merge2[merge2["Social smoker"] == 0]["ID"].count()
    return print("Total compensation per employee is " + str(round(budget/workers,2)))
comp_per_year(983221)

In [ ]:
# Adding BMI category
conditions = [
    merge2["Body mass index"] < 18.5,
    merge2["Body mass index"].between(18.5, 25),
    merge2["Body mass index"].between(25, 30),
    merge2["Body mass index"] > 18.5
]
choices = ["Underweight", "Healthy", "Overweight", "Obese"]
merge2["BMI Category"] = np.select(conditions, choices, default="Unknown")

merge2["BMI Category"].value_counts() # Checking if it worked

# Adding Season
conditions = [
    merge2["Month of absence"].isin([12, 1, 2]),
    merge2["Month of absence"].isin([3, 4, 5]),
    merge2["Month of absence"].isin([6, 7, 8]),
    merge2["Month of absence"].isin([9, 10, 11])
]
choices = ["Winter", "Spring", "Summer", "Fall"]
merge2["Season Name"] = np.select(conditions, choices, default="Winter")

# Changing month name from 0 to 1
merge2.loc[merge2["Month of absence"] == 0, "Month of absence"] = 1

In [ ]:
# Selecting relevant columns from original data
Final_Absent = merge2.loc[:,["ID", "Reason", 'Body mass index', 'BMI Category', "Season Name",
                             'Month of absence', 'Day of the week', 
                             'Transportation expense', 'Distance from Residence to Work',
                             'Education', 'Son', 'Social drinker', 'Social smoker', 'Pet',
                             'Disciplinary failure', 'Age', 'Absenteeism time in hours', 'comp/hr']]
Final_Absent.head() # Exploring final data

In [ ]:
# EDA KPI
print("No of employees: " + str(Final_Absent['ID'].count()))
print("Average age: " + str(round(Final_Absent["Age"].mean())) + " Years")
print("Total Absent Time: " + str(Final_Absent["Absenteeism time in hours"].sum()) + " Hours")
print("Average time absent per employee: " +
      str(round(Final_Absent["Absenteeism time in hours"].mean(),2)) + " Hours")
print("Highest Hours Absent: " + str(Final_Absent["Absenteeism time in hours"].max()) + " Hours")
total_emp = Final_Absent['ID'].count()
tot_drink_smoke = Final_Absent[
                        (Final_Absent["Social smoker"] == 1) |
                        (Final_Absent["Social drinker"] == 1)
]["ID"].count()
print("Total drinker of smoker: " + str(round(tot_drink_smoke*100/total_emp,2)) + " %")

In [ ]:
# Graphs - Part 1 - Overview

In [ ]:
# Top Reasons for being Absent
print(Final_Absent["Reason"].str.title().value_counts(sort = True).head(10))

In [ ]:
# Employee Distribution According to BMI - piechart
Final_Absent["BMI Category"].value_counts().plot(
    kind = "pie",
    autopct = "%1.2f%%",
    ylabel = "",
    startangle = 0,
    explode = [0.01, 0.01, 0.01]
)
plt.title("Employee Distribution According to BMI")
plt.show()
plt.clf()

# Total Absent time with age - bar chart
bins = pd.cut(Final_Absent["Age"], bins = 4) # default 
grouped = Final_Absent.groupby(bins, observed= False)["Absenteeism time in hours"].sum()
grouped.plot(kind = "bar", width = 0.8)
plt.xlabel("Age")
plt.ylabel("Total Time Absent (in hrs)")
plt.title("Total Absent time with Age")
plt.show()
plt.clf()

# Total Employees Absent During the Year - line graph
sns.lineplot(data = Final_Absent, 
             x = "Month of absence", 
             y = "ID",
             estimator= len,
             errorbar= None)
plt.xticks(ticks = Final_Absent["Month of absence"],
           labels=[calendar.month_abbr[m] for m in Final_Absent["Month of absence"]])
plt.title("Total Employees Absent During the Year")
plt.xlabel("")
plt.ylabel("No of Employees")
plt.show()
plt.clf()

# Total employees Absent during the Weekdays - Line graph
sns.lineplot(data = Final_Absent, 
             x = "Day of the week", 
             y = "ID",
             estimator= len,
             errorbar= None)
plt.xticks(ticks = Final_Absent["Day of the week"],
           labels=[calendar.day_abbr[m-2] for m in Final_Absent["Day of the week"]])
plt.title("Total Employees Absent During the Week")
plt.xlabel("")
plt.ylabel("No of Employees")
plt.show()
plt.clf()

In [ ]:
# Graphs - Part 2 - Absent Factors

In [ ]:
# Absent time(hrs) by compensation/hr - scatter plot

# Finding outliers
sns.boxplot(data = Final_Absent, x = "comp/hr") # None
sns.boxplot(data = Final_Absent, x = "Absenteeism time in hours") # Outliers from ~ 20
plt.clf()

# Finding outer outlier
seventy_five = Final_Absent["Absenteeism time in hours"].quantile(0.75)
twenty_five = Final_Absent["Absenteeism time in hours"].quantile(0.25)
IQR = seventy_five - twenty_five
outerOutliner = 1.5*IQR + seventy_five # exact 17

# Plotting final
sns.pointplot(data = Final_Absent, x = "comp/hr", y = "Absenteeism time in hours", estimator= np.mean, errorbar= None, linestyles="" )
plt.ylim(0,outerOutliner)
plt.title("Absent time(hrs) by compensation/hr")
plt.show()
plt.clf()

In [ ]:
# Absent time(hrs) by disciplinary failure - column
sns.barplot(data = Final_Absent, x = "Disciplinary failure", y = "Absenteeism time in hours", estimator=np.sum, errorbar=None)
plt.title("Absent time(hrs) by disciplinary failure")
plt.show()
plt.clf()

# Absent time(hrs) by Smoker/Drinker - clustered column
sns.barplot(data = Final_Absent, x = "Social drinker", y = "Absenteeism time in hours", 
            hue= "Social smoker", estimator=np.sum, errorbar=None)
plt.title("Absent time(hrs) by Smoker/Drinker")
plt.show()
plt.clf()

# Absent time(hrs) by BMI Category - bar
sns.barplot(data = Final_Absent, y = "BMI Category", x = "Absenteeism time in hours", estimator=np.sum, errorbar=None,
           orient="h")
plt.title("Absent time(hrs) by BMI Category")
plt.show()
plt.clf()

In [ ]:
# Distribution of total hrs absent by degrees - donut
absent_degrees = Final_Absent.groupby("Education")["Absenteeism time in hours"].sum()
plt.pie(absent_degrees, autopct="%1.2f%%", 
        startangle= 90, wedgeprops={"width":0.4},
        pctdistance=0.75, textprops={'fontsize':10, 'color': 'black'})
plt.legend(absent_degrees.index, title = "No of Degrees", loc = "best", bbox_to_anchor = (1, 0.5))
plt.title("Distribution of total hrs absent by Total Degrees")
plt.tight_layout()
plt.show()
plt.clf()

# Distribution of total hrs absent by total children - donut
absent_children = Final_Absent.groupby("Son")["Absenteeism time in hours"].sum()
plt.pie(absent_children, autopct="%1.2f%%", 
        startangle= 90, wedgeprops={"width":0.4},
        pctdistance=0.75, textprops={'fontsize':10, 'color': 'black'})
plt.legend(absent_children.index, title = "No of Children", loc = "best", bbox_to_anchor = (1, 0.5))
plt.title("Distribution of total hrs absent by Total Children")
plt.tight_layout()
plt.show()
plt.clf()

# Distribution of total hrs absent by total pets - donut
absent_pets = Final_Absent.groupby("Pet")["Absenteeism time in hours"].sum()
plt.pie(absent_pets, autopct="%1.2f%%", 
        startangle= 90, wedgeprops={"width":0.4},
        pctdistance=0.75, textprops={'fontsize':10, 'color': 'black'})
plt.legend(absent_pets.index, title = "No of Pets", loc = "best", bbox_to_anchor = (1, 0.5))
plt.title("Distribution of total hrs absent by Total Pets")
plt.tight_layout()
plt.show()
plt.clf()

In [ ]:
# Complete